# BlackPearl Morphix — Stage 2, Step 4 (v2): Knockout-conditioned tissue generation

**Question:** given which gene is knocked out in a tumour cell, can the model generate the tissue neighbourhood around it, including the knockout-specific changes found in Step 3 (e.g. NFKBIA cells sitting further from T cells)?

**Data:** Perturb-FISH tumour xenograft (Binan et al., 2025, *Cell*), linked in Step 2. Each neighbourhood = the 63 nearest cells around a focal tumour cell, with positions relative to it and 3 classes: tumour-like, T cell, host.

**Model:** the Stage 1 flow-matching network + a knockout embedding. Condition 0 = "unknown": ~40k unlabelled tumour-like cells teach general tissue structure, and 10% of labelled examples have their knockout hidden, so one network is both the conditional and the unconditional model.

**Evaluation (on held-out tissue blocks):** for each knockout, the predicted shift vs Control is compared with the shift actually observed in held-out tissue, against:
- **No-effect baseline:** predicts zero shift (what an unconditional model implies).
- **Lookup baseline:** predicts the shift measured in training tissue (a strong, simple reference).
- **Noise floor:** the error you'd get even with perfect predictions, because held-out groups are small.

**Changes from v1:** (1) the focal cell is now an explicit fixed token at the centre, so generated neighbours keep a realistic distance from it; (2) fair ablation: held-out loss with the **true knockout vs a Control label** on the same cells; (3) **location-matched** evaluation: each knockout is compared with Control cells from the same tissue block, as in the Step 3 signal check (v1 showed raw knockout-vs-Control differences are dominated by *where* each knockout's cells sit).

Runtime: **GPU (T4)**. ~20–25 min. You'll be asked to upload `morphix_pfish_cells (1).csv.gz`. At the end, copy the **SUMMARY** block and paste it back.

In [ ]:
import os, json, time, copy, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then restart.')

CFG = dict(n_neighbours=63, test_frac=0.2, n_blocks=6, seed=0, min_cells_per_gene=30, max_unlabelled=40000,
           d_model=128, n_layers=4, n_heads=4, batch_size=128, steps=15000, lr=3e-4, warmup=500, ema=0.999,
           p_drop=0.1, frac_labelled_per_batch=0.5, n_gen=400, sample_steps=100, guidance=1.0)
rng = np.random.default_rng(CFG['seed']); torch.manual_seed(CFG['seed'])
os.makedirs('results_s2', exist_ok=True)
S = {'config': CFG}

## 1. Data

In [ ]:
# ---------- DATA: focal tumour cells and their 63-cell neighbourhoods ----------
import os, glob
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

TYPES = ['tumour_like', 'tcell', 'host']          # neighbour classes generated by the model


def load_cells():
    cands = sorted(glob.glob('morphix_pfish_cells*.csv.gz'), key=os.path.getmtime)
    if not cands:
        from google.colab import files
        print('Upload the morphix_pfish_cells file from the SUCCESSFUL Step 2 run '
              '(it may be named "morphix_pfish_cells (1).csv.gz"):')
        up = files.upload()
        cands = [n for n in up if n.startswith('morphix_pfish_cells')]
    fn = cands[-1]
    cells = pd.read_csv(fn, index_col='cell_id')
    cells['gene'] = cells['gene'].fillna('').astype(str)
    if not (cells.cell_class == 'tumour').any():
        raise ValueError(f'{fn} has no tumour cells (old file from the failed run). Delete it and upload the newer one.')
    print('using', fn)
    return cells


def assign_classes(cells):
    """Same rule as the Step 3 signal check: 'other' cells with tumour-like counts are tumour-like,
    low-count 'other' cells (mostly mouse host) are 'host'."""
    tc = np.log10(cells['total_counts'] + 1)
    thr = np.quantile(tc[cells.cell_class == 'tumour'], 0.05)
    cls = np.where(cells.cell_class == 'tcell', 'tcell',
                   np.where((cells.cell_class == 'tumour') | (tc >= thr), 'tumour_like', 'host'))
    cells['ntype'] = pd.Categorical(cls, categories=TYPES).codes.astype(np.int64)
    return thr


def block_split_xy(xy, test_frac, n_blocks, seed):
    rng = np.random.default_rng(seed)
    lo, hi = xy.min(0), xy.max(0)
    b = np.minimum(((xy - lo) / (hi - lo + 1e-9) * n_blocks).astype(int), n_blocks - 1)
    bid = b[:, 0] * n_blocks + b[:, 1]
    counts = np.bincount(bid, minlength=n_blocks ** 2)
    test, n = [], 0
    for i in rng.permutation(np.unique(bid)):
        if n >= test_frac * len(xy):
            break
        test.append(i); n += counts[i]
    return np.isin(bid, test)


def build_patches(cells, cfg, rng):
    XY = cells[['x', 'y']].values
    tree = cKDTree(XY)
    d2, _ = tree.query(XY, k=2)
    unit = float(np.median(d2[:, 1]))
    n_nb = cfg['n_neighbours']
    is_test = block_split_xy(XY, cfg['test_frac'], cfg['n_blocks'], cfg['seed'])

    # conditions: 0 = unknown (unlabelled tumour-like cells), 1 = Control, 2.. = knockouts with enough cells
    tum = cells[(cells.cell_class == 'tumour') & ~cells.gene.isin(['multiple', 'none', ''])]
    vc = tum.gene.value_counts()
    genes = [g for g, n in vc.items() if g != 'Control' and n >= cfg['min_cells_per_gene']]
    cond_names = ['unknown', 'Control'] + sorted(genes)
    cond_id = {g: i for i, g in enumerate(cond_names)}
    cond = np.full(len(cells), -1)
    labelled = (cells.cell_class == 'tumour').values & cells.gene.isin(cond_names[1:]).values
    cond[labelled] = [cond_id[g] for g in cells.gene.values[labelled]]
    unlab = (cells.cell_class == 'other').values & (cells.ntype.values == TYPES.index('tumour_like'))
    cond[unlab] = 0

    focal = np.flatnonzero(cond >= 0)
    d, nb = tree.query(XY[focal], k=n_nb + 1)
    nb = nb[:, 1:]
    clean = (is_test[nb] == is_test[focal][:, None]).all(1)       # patch entirely inside its split
    focal, nb = focal[clean], nb[clean]
    pos = ((XY[nb] - XY[focal][:, None, :]) / unit).astype(np.float32)   # neighbours relative to the focal cell
    typ = cells.ntype.values[nb].astype(np.int64)
    c = cond[focal]
    te = is_test[focal]

    # cap the number of unlabelled patches (they teach general structure; labelled ones teach knockout effects)
    keep = np.ones(len(focal), bool)
    for split in [False, True]:
        idx = np.flatnonzero((c == 0) & (te == split))
        cap = cfg['max_unlabelled'] if not split else cfg['max_unlabelled'] // 4
        if len(idx) > cap:
            keep[rng.choice(idx, len(idx) - cap, replace=False)] = False
    data = dict(pos=pos[keep], typ=typ[keep], cond=c[keep], test=te[keep], focal=focal[keep])
    return data, cond_names, unit

In [ ]:
cells = load_cells()
thr = assign_classes(cells)
data, COND, UNIT = build_patches(cells, CFG, rng)
K = len(TYPES)
tr, te = ~data['test'], data['test']
lab = data['cond'] >= 1
print(f'cell spacing unit {UNIT:.1f} | conditions: {len(COND)} ({len(COND) - 2} knockouts + unknown + Control)')
print(f'train: {(tr & lab).sum()} labelled + {(tr & ~lab).sum()} unlabelled | test: {(te & lab).sum()} labelled')
print('test cells per condition:', pd.Series(data['cond'][te & lab]).map(lambda i: COND[i]).value_counts().to_dict())
POS_SCALE = float(data['pos'][tr].std())
S.update(n_train_labelled=int((tr & lab).sum()), n_train_unlabelled=int((tr & ~lab).sum()),
         n_test_labelled=int((te & lab).sum()), conditions=COND, unit=UNIT, pos_scale=POS_SCALE)

## 2. Model

In [ ]:
# ---------- MODEL: knockout-conditioned flow matching over a cell neighbourhood ----------
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def pairwise_dist(p):
    diff = p[:, :, None, :] - p[:, None, :, :]
    return torch.sqrt((diff ** 2).sum(-1) + 1e-8)


class TimeEmbedding(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))

    def forward(self, t):
        half = self.d // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        a = 1000.0 * t[:, None] * freqs[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], dim=-1))


class DistanceBias(nn.Module):
    def __init__(self, n_heads, n_rbf=16, max_dist=4.0):
        super().__init__()
        self.register_buffer('centres', torch.linspace(0, max_dist, n_rbf))
        self.gamma = (n_rbf / max_dist) ** 2
        self.proj = nn.Linear(n_rbf, n_heads)

    def forward(self, pos):
        d = pairwise_dist(pos)
        rbf = torch.exp(-self.gamma * (d[..., None] - self.centres) ** 2)
        return self.proj(rbf).permute(0, 3, 1, 2)


class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.bias = DistanceBias(n_heads)
        self.ada = nn.Linear(d, 6 * d)
        nn.init.zeros_(self.ada.weight)
        nn.init.zeros_(self.ada.bias)

    def forward(self, h, c, pos):
        B, N, _ = h.shape
        s1, b1, g1, s2, b2, g2 = self.ada(c)[:, None].chunk(6, dim=-1)
        mask = self.bias(pos).reshape(B * self.n_heads, N, N)
        x = self.norm1(h) * (1 + s1) + b1
        h = h + g1 * self.attn(x, x, x, attn_mask=mask, need_weights=False)[0]
        x = self.norm2(h) * (1 + s2) + b2
        return h + g2 * self.mlp(x)


class ConditionalTissueFlow(nn.Module):
    """v_theta(x_t, t | knockout). The focal (knocked-out) cell is a fixed token at the origin; the 63 neighbours are
    generated around it. The knockout enters through a learned embedding added to the time embedding,
    so it modulates every block. Condition 0 = 'unknown', which doubles as the unconditional model."""
    def __init__(self, n_types, n_cond, d=128, n_layers=4, n_heads=4):
        super().__init__()
        self.dim = 2 + n_types
        self.inp = nn.Linear(self.dim, d)
        self.time = TimeEmbedding(d)
        self.cond = nn.Embedding(n_cond, d)
        nn.init.normal_(self.cond.weight, std=0.02)
        self.focal = nn.Parameter(torch.randn(1, 1, d) * 0.02)   # fixed token for the focal cell at the origin
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.out_norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, self.dim)

    def forward(self, x, t, cond):
        B = x.shape[0]
        c = self.time(t) + self.cond(cond)
        # prepend the focal cell as a fixed token at (0, 0): neighbours can 'see' it and keep a realistic distance
        h = torch.cat([self.focal.expand(B, 1, -1), self.inp(x)], dim=1) + c[:, None]
        pos = torch.cat([torch.zeros(B, 1, 2, device=x.device, dtype=x.dtype), x[..., :2]], dim=1)
        for blk in self.blocks:
            h = blk(h, c, pos)
        return self.out(self.out_norm(h))[:, 1:]               # velocities for the 63 neighbours only


def random_rotate(pos):
    """Rotate/reflect neighbourhoods about the focal cell (origin)."""
    B = pos.shape[0]
    th = torch.rand(B, device=pos.device) * 2 * math.pi
    c, s = th.cos(), th.sin()
    R = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)
    pos = pos @ R.transpose(1, 2)
    flip = (torch.rand(B, 1, device=pos.device) < 0.5).float() * 2 - 1
    return torch.cat([pos[..., :1] * flip[..., None], pos[..., 1:]], dim=-1)


def cond_flow_loss(model, pos, typ, cond, n_types, pos_scale, p_drop=0.1):
    B = pos.shape[0]
    x1 = torch.cat([pos / pos_scale, F.one_hot(typ, n_types).float()], dim=-1)
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=pos.device)
    tt = t[:, None, None]
    xt = (1 - tt) * x0 + tt * x1
    drop = torch.rand(B, device=pos.device) < p_drop          # hide the knockout -> learn the unconditional model too
    cond = torch.where(drop, torch.zeros_like(cond), cond)
    v = model(xt, t, cond)
    err = (v - (x1 - x0)) ** 2
    lp, lt = err[..., :2].mean(), err[..., 2:].mean()
    return lp + lt, {'pos': lp.item(), 'type': lt.item()}


@torch.no_grad()
def sample_cond(model, cond_id, n, n_cells, n_types, pos_scale, steps=100, batch=250, guidance=1.0, device='cpu', seed=0):
    """Generate n neighbourhoods for one condition. guidance=1: plain conditional model;
    guidance>1: classifier-free guidance (exaggerates the knockout-specific signal)."""
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    P, T = [], []
    for i in range(0, n, batch):
        m = min(batch, n - i)
        x = torch.randn(m, n_cells, 2 + n_types, device=device, generator=g)
        c = torch.full((m,), cond_id, device=device, dtype=torch.long)
        c0 = torch.zeros_like(c)
        for k in range(steps):
            t = torch.full((m,), k / steps, device=device)
            v = model(x, t, c)
            if guidance != 1.0:
                v0 = model(x, t, c0)
                v = v0 + guidance * (v - v0)
            x = x + v / steps
        P.append((x[..., :2] * pos_scale).cpu().numpy())
        T.append(x[..., 2:].argmax(-1).cpu().numpy())
    model.train()
    return np.concatenate(P), np.concatenate(T)

## 3. Training

In [ ]:
P = torch.tensor(data['pos'], device=DEVICE); T_ = torch.tensor(data['typ'], device=DEVICE)
C = torch.tensor(data['cond'], device=DEVICE)
idx_lab = torch.tensor(np.flatnonzero(tr & lab), device=DEVICE)
idx_unl = torch.tensor(np.flatnonzero(tr & ~lab), device=DEVICE)
idx_test = torch.tensor(np.flatnonzero(te & lab), device=DEVICE)
idx_test_ko = torch.tensor(np.flatnonzero(te & (data['cond'] >= 2)), device=DEVICE)

model = ConditionalTissueFlow(K, len(COND), CFG['d_model'], CFG['n_layers'], CFG['n_heads']).to(DEVICE)
ema = copy.deepcopy(model).eval()
for p in ema.parameters(): p.requires_grad_(False)
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / CFG['warmup']) *
                                          0.5 * (1 + math.cos(math.pi * min(1.0, s / CFG['steps']))))
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')
n_lab = int(CFG['batch_size'] * CFG['frac_labelled_per_batch'])
log, t0 = [], time.time()
for step in range(CFG['steps'] + 1):
    b = torch.cat([idx_lab[torch.randint(0, len(idx_lab), (n_lab,), device=DEVICE)],
                   idx_unl[torch.randint(0, len(idx_unl), (CFG['batch_size'] - n_lab,), device=DEVICE)]])
    loss, parts = cond_flow_loss(model, random_rotate(P[b]), T_[b], C[b], K, POS_SCALE, CFG['p_drop'])
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    with torch.no_grad():
        for pe, p in zip(ema.parameters(), model.parameters()): pe.lerp_(p, 1 - CFG['ema'])
    if step % 1000 == 0:
        with torch.no_grad():
            torch.manual_seed(123)
            j = idx_test[torch.randint(0, len(idx_test), (512,), device=DEVICE)]
            jk = idx_test_ko[torch.randint(0, len(idx_test_ko), (512,), device=DEVICE)]   # knockout cells only
            torch.manual_seed(123)
            vl_c, _ = cond_flow_loss(ema, P[jk], T_[jk], C[jk], K, POS_SCALE, 0.0)                       # true knockout
            torch.manual_seed(123)
            vl_k, _ = cond_flow_loss(ema, P[jk], T_[jk], torch.full_like(C[jk], 1), K, POS_SCALE, 0.0)    # labelled 'Control'
            torch.manual_seed(123)
            vl_u, _ = cond_flow_loss(ema, P[jk], T_[jk], torch.zeros_like(C[jk]), K, POS_SCALE, 0.0)     # 'unknown'
            torch.manual_seed(CFG['seed'] + step)
        log.append(dict(step=step, train=loss.item(), test_true=vl_c.item(), test_as_control=vl_k.item(), test_unknown=vl_u.item()))
        print(f'step {step:6d} | train {loss.item():.4f} | held-out knockout cells: true label {vl_c.item():.4f}, '
              f'as Control {vl_k.item():.4f}, unknown {vl_u.item():.4f} | {time.time() - t0:.0f}s')
log = pd.DataFrame(log)
torch.save({'model': ema.state_dict(), 'cfg': CFG, 'conditions': COND, 'pos_scale': POS_SCALE, 'unit': UNIT},
           'results_s2/morphix_stage2.pt')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(log.step, log.test_true, 'o-', label='true knockout label')
ax.plot(log.step, log.test_as_control, 's--', label="labelled as 'Control'")
ax.plot(log.step, log.test_unknown, '^:', label="labelled 'unknown'")
ax.set_xlabel('step'); ax.set_ylabel('held-out loss on knockout cells'); ax.legend(); ax.set_title('Does knowing the knockout help?')
plt.tight_layout(); plt.savefig('results_s2/training.png', dpi=150); plt.show()
S['heldout_loss_knockout_cells'] = {'true_label': float(log.test_true.iloc[-1]),
                                     'as_Control': float(log.test_as_control.iloc[-1]),
                                     'unknown': float(log.test_unknown.iloc[-1])}
S['heldout_loss_true_minus_control_last3_avg'] = float((log.test_true - log.test_as_control).iloc[-3:].mean())

## 4. Generate neighbourhoods for every condition

In [ ]:
gen = {}
t0 = time.time()
for cid in range(len(COND)):
    gp, gt = sample_cond(ema, cid, CFG['n_gen'], CFG['n_neighbours'], K, POS_SCALE, CFG['sample_steps'],
                         guidance=CFG['guidance'], device=DEVICE, seed=cid)
    gen[cid] = (gp, gt)
print(f'generated {CFG["n_gen"]} neighbourhoods x {len(COND)} conditions in {time.time() - t0:.0f}s')

## 5. Evaluation

In [ ]:
# ---------- EVALUATION: does the model reproduce knockout effects on held-out tissue? ----------
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance, pearsonr

FEATS = ['frac_tcell', 'frac_host', 'mean_dist', 'nn_dist', 'dist_to_tcell']


def patch_features(pos, typ):
    """Per-neighbourhood summaries, all relative to the focal cell at the origin (cell-spacing units)."""
    r = np.sqrt((pos ** 2).sum(-1))                      # distance of each neighbour to the focal cell
    is_t = typ == 1
    dt = np.where(is_t, r, np.inf).min(1)
    dt = np.where(np.isfinite(dt), dt, r.max(1))         # no T cell in the patch -> patch edge
    return pd.DataFrame({'frac_tcell': is_t.mean(1), 'frac_host': (typ == 2).mean(1),
                         'mean_dist': r.mean(1), 'nn_dist': r.min(1), 'dist_to_tcell': dt})


def shifts(feat_df, cond, genes_ids, control_id):
    """Mean feature of each knockout minus mean of Control (same set of cells)."""
    ctrl = feat_df[cond == control_id]
    out = {}
    for g in genes_ids:
        sel = feat_df[cond == g]
        out[g] = (sel.mean() - ctrl.mean()).values, len(sel)
    return out, ctrl


def matched_shifts(feat_df, cond, block, genes_ids, control_id):
    """Location-matched effect: for each knockout, compare its cells with Control cells from the SAME tissue
    block (as in the Step 3 signal check), then average over blocks weighted by the knockout's cells."""
    X = feat_df.values
    out = {}
    ctrl = cond == control_id
    for g in genes_ids:
        isg = cond == g
        diffs, w, var_g, var_c = [], [], [], []
        for b in np.unique(block[isg]):
            gi, ci = isg & (block == b), ctrl & (block == b)
            if ci.sum() == 0:
                continue
            diffs.append(X[gi].mean(0) - X[ci].mean(0)); w.append(gi.sum())
            var_c.append(X[ci].var(0) / ci.sum() * gi.sum() ** 2)
        if not w:
            out[g] = (np.full(X.shape[1], np.nan), 0, np.full(X.shape[1], np.nan)); continue
        w = np.array(w, float); diffs = np.array(diffs)
        n_used = int(w.sum())
        sel = X[isg & np.isin(block, [b for b in np.unique(block[isg]) if (ctrl & (block == b)).any()])]
        se = np.sqrt(sel.var(0) / n_used + np.sum(var_c, 0) / w.sum() ** 2)
        out[g] = ((w[:, None] * diffs).sum(0) / w.sum(), n_used, se)
    return out


def evaluate_matched(train_df, train_c, train_b, test_df, test_c, test_b, gen, cond_names, min_test=10):
    ctrl_id = cond_names.index('Control')
    genes = [i for i in range(2, len(cond_names)) if (test_c == i).sum() >= min_test and (train_c == i).sum() >= min_test]
    te = matched_shifts(test_df, test_c, test_b, genes, ctrl_id)
    trn = matched_shifts(train_df, train_c, train_b, genes, ctrl_id)
    sd = test_df[test_c == ctrl_id].std().values + 1e-9
    rows = []
    for g in genes:
        real, n_g, se = te[g]
        if n_g < min_test or not np.all(np.isfinite(trn[g][0])):
            continue
        model = (gen[g].mean() - gen[ctrl_id].mean()).values
        for j, f in enumerate(FEATS):
            rows.append(dict(gene=cond_names[g], feature=f, n_test_matched=n_g, real_test=real[j] / sd[j],
                             test_se=se[j] / sd[j], lookup_train=trn[g][0][j] / sd[j], model=model[j] / sd[j]))
    tab = pd.DataFrame(rows)
    y = tab.real_test.values
    summ = {'n_knockouts_evaluated': int(tab.gene.nunique()), 'n_gene_feature_pairs': len(tab),
            'noise_floor_mse': float(np.mean(tab.test_se ** 2)),
            'mse_zero': float(np.mean(y ** 2)),
            'mse_lookup': float(np.mean((tab.lookup_train - y) ** 2)),
            'mse_model': float(np.mean((tab.model - y) ** 2)),
            'r_lookup_vs_test': float(pearsonr(tab.lookup_train, y)[0]),
            'r_model_vs_test': float(pearsonr(tab.model, y)[0]),
            'r_model_vs_train': float(pearsonr(tab.model, tab.lookup_train)[0])}
    return tab, summ


def evaluate_effects(train_df, train_c, test_df, test_c, gen, cond_names, min_test=10):
    """gen: {cond_id: features DataFrame of generated neighbourhoods}. Returns long table + summary."""
    ctrl_id = cond_names.index('Control')
    genes = [i for i in range(2, len(cond_names)) if (test_c == i).sum() >= min_test and (train_c == i).sum() >= min_test]
    test_sh, test_ctrl = shifts(test_df, test_c, genes, ctrl_id)
    train_sh, _ = shifts(train_df, train_c, genes, ctrl_id)
    sd = test_ctrl.std().values + 1e-9
    rows = []
    for g in genes:
        real, n_g = test_sh[g]
        sel = test_df[test_c == g]
        se = np.sqrt(sel.var().values / n_g + test_ctrl.var().values / len(test_ctrl)) / sd
        model = (gen[g].mean() - gen[ctrl_id].mean()).values
        for j, f in enumerate(FEATS):
            rows.append(dict(gene=cond_names[g], feature=f, n_test=n_g,
                             real_test=real[j] / sd[j], test_se=se[j],
                             lookup_train=train_sh[g][0][j] / sd[j], model=model[j] / sd[j]))
    tab = pd.DataFrame(rows)
    y = tab.real_test.values
    summ = {
        'n_knockouts_evaluated': len(genes), 'n_gene_feature_pairs': len(tab),
        'noise_floor_mse (unavoidable, from test sampling error)': float(np.mean(tab.test_se ** 2)),
        'mse_zero (no-effect baseline)': float(np.mean(y ** 2)),
        'mse_lookup (train shifts as prediction)': float(np.mean((tab.lookup_train - y) ** 2)),
        'mse_model': float(np.mean((tab.model - y) ** 2)),
        'r_lookup_vs_test': float(pearsonr(tab.lookup_train, y)[0]),
        'r_model_vs_test': float(pearsonr(tab.model, y)[0]),
        'r_model_vs_train (did it learn the training effects?)': float(pearsonr(tab.model, tab.lookup_train)[0]),
    }
    return tab, summ


def realism(gen_df, test_df, train_df):
    """Per-feature W1 distance to held-out Control neighbourhoods, vs. real training Control (reference)."""
    return pd.DataFrame({'generated vs test': [wasserstein_distance(gen_df[f], test_df[f]) for f in FEATS],
                         'real train vs test (reference)': [wasserstein_distance(train_df[f], test_df[f]) for f in FEATS]},
                        index=FEATS)

In [ ]:
F_all = patch_features(data['pos'], data['typ'])
trl, tel = tr & lab, te & lab
F_tr, c_tr = F_all[trl].reset_index(drop=True), data['cond'][trl]
F_te, c_te = F_all[tel].reset_index(drop=True), data['cond'][tel]
F_gen = {cid: patch_features(*gen[cid]) for cid in gen}

tab_raw, summ_raw = evaluate_effects(F_tr, c_tr, F_te, c_te, F_gen, COND)
print('--- RAW shifts (v1 evaluation; dominated by location) ---')
for k, v in summ_raw.items(): print(f'  {k:58s} {v:.4f}' if isinstance(v, float) else f'  {k:58s} {v}')
S['effects_raw'] = summ_raw

BLOCK = 40 * UNIT
fx, fy = cells.x.values[data['focal']], cells.y.values[data['focal']]
blk = (np.floor(fx / BLOCK).astype(int) * 100000 + np.floor(fy / BLOCK).astype(int))
tab, summ = evaluate_matched(F_tr, c_tr, blk[trl], F_te, c_te, blk[tel], F_gen, COND)
print('\n--- LOCATION-MATCHED shifts (knockout vs Control in the same tissue block) ---')
for k, v in summ.items(): print(f'  {k:58s} {v:.4f}' if isinstance(v, float) else f'  {k:58s} {v}')
S['effects_matched'] = summ
tab.to_csv('results_s2/effects_matched.csv', index=False); tab_raw.to_csv('results_s2/effects_raw.csv', index=False)

ci = COND.index('Control')
real = realism(F_gen[ci], F_te[c_te == ci], F_tr[c_tr == ci])
print('\nRealism of generated Control neighbourhoods (W1 distance to held-out Control; lower = better):')
print(real.round(4))
S['realism_control'] = real.round(4).to_dict()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
lim = np.nanmax(np.abs(tab[['real_test', 'model', 'lookup_train']].values)) * 1.1
for a, col, name in [(ax[0], 'model', 'Morphix (conditioned model)'), (ax[1], 'lookup_train', 'Lookup baseline (train shifts)')]:
    a.errorbar(tab[col], tab.real_test, yerr=tab.test_se, fmt='o', alpha=0.5, ms=4)
    a.plot([-lim, lim], [-lim, lim], 'k--', lw=1); a.axhline(0, c='grey', lw=0.5); a.axvline(0, c='grey', lw=0.5)
    a.set_xlim(-lim, lim); a.set_ylim(-lim, lim)
    a.set_xlabel(f'predicted shift vs Control (SD units)'); a.set_ylabel('observed shift in held-out tissue')
    r = np.corrcoef(tab[col], tab.real_test)[0, 1]
    a.set_title(f'{name} (location-matched)\nr = {r:.2f}')
plt.tight_layout(); plt.savefig('results_s2/predicted_vs_observed.png', dpi=150); plt.show()

# Focus on the strongest Step-3 effects
focus = [('NFKBIA', 'dist_to_tcell'), ('NFKBIA', 'frac_tcell'), ('IRF7', 'frac_tcell'), ('LBP', 'frac_host'),
         ('IRF5', 'frac_host'), ('MAP2K6', 'frac_host'), ('MAP2K3', 'frac_tcell')]
ft = tab.set_index(['gene', 'feature'])
rows = [dict(effect=f'{g} {f}', **ft.loc[(g, f), ['real_test', 'test_se', 'lookup_train', 'model']].to_dict())
        for g, f in focus if (g, f) in ft.index]
focus_df = pd.DataFrame(rows)
print('Strongest Step-3 effects (SD units vs Control):'); print(focus_df.round(3).to_string(index=False))
S['focus_effects'] = focus_df.round(3).to_dict(orient='records')

In [ ]:
# Example neighbourhoods: real vs generated, Control vs NFKBIA (if available)
COL = np.array(['#d62728', '#1f77b4', '#bbbbbb'])
show = [c for c in ['Control', 'NFKBIA'] if c in COND]
fig, axes = plt.subplots(2 * len(show), 4, figsize=(12, 6 * len(show)))
for si, gname in enumerate(show):
    cid = COND.index(gname)
    real_idx = np.flatnonzero(tel & (data['cond'] == cid))[:4]
    for j in range(4):
        for row, (pp, tt, lbl) in enumerate([(data['pos'][real_idx[j]], data['typ'][real_idx[j]], 'real (held-out)'),
                                             (gen[cid][0][j], gen[cid][1][j], 'generated')]):
            a = axes[2 * si + row, j]
            a.scatter(pp[:, 0], pp[:, 1], c=COL[tt], s=35, edgecolors='k', linewidths=0.3)
            a.scatter([0], [0], marker='*', s=200, c='gold', edgecolors='k')
            a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([]); a.set_xlim(-6, 6); a.set_ylim(-6, 6)
            if j == 0: a.set_ylabel(f'{gname}\n{lbl}')
fig.suptitle('Neighbourhoods around a knocked-out tumour cell (gold star). Red: tumour-like, blue: T cell, grey: host')
plt.tight_layout(); plt.savefig('results_s2/examples.png', dpi=150); plt.show()

!zip -qr morphix_stage2_results.zip results_s2
try:
    from google.colab import files; files.download('morphix_stage2_results.zip')
except Exception:
    pass

## How to read the result
- **Model MSE well below the no-effect baseline** → the knockout conditioning carries real, generalisable information.
- **Model close to the lookup baseline** → the model has learned the knockout effects about as well as the training data allows.
- **Everything near the noise floor** → held-out groups are too small to tell methods apart; more data (more cells per knockout) is needed. That is the data bottleneck that motivates purpose-built, automated experiments.

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(S, indent=1, default=str)[:15000])
print('===== SUMMARY END =====')